# Distributed Training

## Learning Objectives
1. Simulate data-parallel gradient averaging across two workers using NumPy.
2. Mock DDP training in PyTorch with gradient synchronisation and OOM handling.
3. Use gradient accumulation to simulate large effective batch sizes.
4. Apply automatic mixed precision (AMP) and measure memory and speed improvement.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import copy
import time

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Data-Parallel Gradient Averaging (NumPy)

In [ ]:
# Simulate 2-worker data-parallel training:
# each worker computes gradients on its shard, then AllReduce averages them.

class LinearModelNumpy:
    def __init__(self, d_in, d_out, seed=0):
        rng = np.random.default_rng(seed)
        self.W = rng.standard_normal((d_in, d_out)) * 0.1
        self.b = np.zeros(d_out)

    def forward(self, X):
        return X @ self.W + self.b

    def mse_grad(self, X, y):
        # Return gradients of MSE loss w.r.t. W and b.
        pred = self.forward(X)
        err  = pred - y                       # (n, d_out)
        dW   = (2 / len(X)) * X.T @ err      # (d_in, d_out)
        db   = (2 / len(X)) * err.sum(0)     # (d_out,)
        return dW, db

N_ddp_np = 200; D_IN, D_OUT = 5, 1
X_dist = np.random.randn(N_ddp_np, D_IN)
W_true = np.array([1, -2, 0.5, 3, -1]).reshape(D_IN, D_OUT)
y_dist = X_dist @ W_true + 0.1 * np.random.randn(N_ddp_np, D_OUT)

model_np = LinearModelNumpy(D_IN, D_OUT)
X_w1, y_w1 = X_dist[:N_ddp_np//2], y_dist[:N_ddp_np//2]
X_w2, y_w2 = X_dist[N_ddp_np//2:], y_dist[N_ddp_np//2:]

LR = 0.05
for step in range(50):
    dW1, db1 = model_np.mse_grad(X_w1, y_w1)
    dW2, db2 = model_np.mse_grad(X_w2, y_w2)
    # AllReduce: average gradients from both workers
    model_np.W -= LR * (dW1 + dW2) / 2
    model_np.b -= LR * (db1 + db2) / 2

mse_final = ((model_np.forward(X_dist) - y_dist) ** 2).mean()
print(f"Data-parallel training MSE after 50 steps: {mse_final:.6f}")
print(f"Learned W: {model_np.W.flatten().round(3)}")
print(f"True    W: {W_true.flatten()}")


## Level 2: Mock DDP with Gradient Sync (PyTorch)

In [ ]:
# Two model replicas, each processing its shard, gradients averaged before update.

def build_model_ddp():
    return nn.Sequential(
        nn.Linear(10, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1)
    ).to(device)

def sync_gradients(models):
    # Average gradients across all model replicas (mock AllReduce).
    n = len(models)
    for param_idx, params in enumerate(zip(*[list(m.parameters()) for m in models])):
        grad_avg = sum(p.grad for p in params if p.grad is not None)
        if grad_avg is not None:
            grad_avg = grad_avg / n
            for p in params:
                if p.grad is not None:
                    p.grad.copy_(grad_avg)

N_ddp = 400
X_ddp = torch.randn(N_ddp, 10, device=device)
y_ddp = (X_ddp[:, 0] + X_ddp[:, 1]).unsqueeze(1)
model1, model2 = build_model_ddp(), build_model_ddp()
model2.load_state_dict(model1.state_dict())   # same initialisation (DDP requirement)
opt1 = torch.optim.Adam(model1.parameters(), lr=1e-3)
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-3)
crit_ddp = nn.MSELoss()

for step in range(60):
    opt1.zero_grad(); opt2.zero_grad()
    try:
        loss1 = crit_ddp(model1(X_ddp[:N_ddp//2]), y_ddp[:N_ddp//2])
        loss2 = crit_ddp(model2(X_ddp[N_ddp//2:]), y_ddp[N_ddp//2:])
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            print("OOM -- reduce batch size or model size")
            torch.cuda.empty_cache()
            continue
        raise
    loss1.backward(); loss2.backward()
    sync_gradients([model1, model2])
    opt1.step(); opt2.step()

with torch.no_grad():
    val_loss = crit_ddp(model1(X_ddp), y_ddp).item()
max_diff = max((p1-p2).abs().max().item()
               for p1, p2 in zip(model1.parameters(), model2.parameters()))
print(f"Mock DDP final val MSE: {val_loss:.6f}")
print(f"Max param diff between replicas: {max_diff:.2e}  (should be ~0 if sync is correct)")


## Real-World Example 1: Gradient Accumulation

In [ ]:
# Gradient accumulation: sum gradients over N micro-batches before stepping.
# Effective batch = micro_batch * ACCUM_STEPS, with the same memory as micro_batch.

N_ga = 800
X_ga = torch.randn(N_ga, 20, device=device)
y_ga = (X_ga[:, :3].sum(1) > 0).long()
ds_ga = TensorDataset(X_ga, y_ga)
ld_ga = DataLoader(ds_ga, batch_size=16, shuffle=True)  # small physical batch

model_ga = nn.Sequential(
    nn.Linear(20, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 2)
).to(device)
opt_ga = torch.optim.Adam(model_ga.parameters(), lr=1e-3)
crit_ga = nn.CrossEntropyLoss()

ACCUM_STEPS = 8   # effective batch = 16 * 8 = 128
step_losses = []
model_ga.train()
opt_ga.zero_grad()

for step, (xb, yb) in enumerate(ld_ga):
    # Divide loss by accumulation steps so gradient magnitude matches a single big batch
    loss = crit_ga(model_ga(xb), yb) / ACCUM_STEPS
    loss.backward()
    step_losses.append(loss.item() * ACCUM_STEPS)
    if (step + 1) % ACCUM_STEPS == 0:
        opt_ga.step()
        opt_ga.zero_grad()

model_ga.eval()
with torch.no_grad():
    acc_ga = (model_ga(X_ga).argmax(1) == y_ga).float().mean().item()
print(f"Gradient accumulation (eff. batch={16*ACCUM_STEPS}) accuracy: {acc_ga:.4f}")
print(f"Mean step loss: {np.mean(step_losses):.4f}")
print(f"Physical batch=16, Effective batch={16*ACCUM_STEPS} (no extra GPU memory)")


## Real-World Example 2: Mixed Precision Training (AMP)

In [ ]:
# AMP: fp16 for compute, fp32 for gradient accumulation.
# ~2x memory reduction, 1.5-3x speedup on CUDA; no change in accuracy.

N_amp = 1000
X_amp = torch.randn(N_amp, 32, device=device)
y_amp = torch.randint(0, 5, (N_amp,), device=device)
ds_amp = TensorDataset(X_amp, y_amp)
ld_amp = DataLoader(ds_amp, batch_size=64, shuffle=True)

model_amp = nn.Sequential(
    nn.Linear(32, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 5)
).to(device)
crit_amp = nn.CrossEntropyLoss()

# --- FP32 baseline ---
model_fp32 = copy.deepcopy(model_amp)
opt_fp32   = torch.optim.Adam(model_fp32.parameters(), lr=1e-3)
t0 = time.perf_counter()
for _ in range(5):
    model_fp32.train()
    for xb, yb in ld_amp:
        opt_fp32.zero_grad()
        crit_amp(model_fp32(xb), yb).backward()
        opt_fp32.step()
t_fp32 = time.perf_counter() - t0

# --- AMP (fp16 on CUDA, transparent fp32 fallback on CPU) ---
model_amp2 = copy.deepcopy(model_amp)
opt_amp2   = torch.optim.Adam(model_amp2.parameters(), lr=1e-3)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
t0 = time.perf_counter()
for _ in range(5):
    model_amp2.train()
    for xb, yb in ld_amp:
        opt_amp2.zero_grad()
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            loss_amp = crit_amp(model_amp2(xb), yb)
        scaler.scale(loss_amp).backward()
        scaler.step(opt_amp2)
        scaler.update()
t_amp = time.perf_counter() - t0

speedup = t_fp32 / max(t_amp, 1e-9)
print(f"FP32 time: {t_fp32*1000:.1f}ms  AMP time: {t_amp*1000:.1f}ms")
print(f"Speedup: {speedup:.2f}x  (full benefit on CUDA; AMP enabled={device.type=='cuda'})")
print("AMP cuts memory ~2x and speeds compute; use for any GPU training run.")


## Real-World Example 3: Gradient Checkpointing

In [ ]:
# Gradient checkpointing: recompute activations during backward instead of storing them.
# Trades ~20-30% extra compute for ~30-50% less memory.

from torch.utils.checkpoint import checkpoint as ckpt_fn

class DeepNetCheckpointed(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(64, 64), nn.ReLU()) for _ in range(8)
        ])
        self.head = nn.Linear(64, 1)

    def forward(self, x, use_ckpt=True):
        for layer in self.layers:
            if use_ckpt and x.requires_grad:
                x = ckpt_fn(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return self.head(x)

X_ckpt = torch.randn(64, 64, device=device)
y_ckpt = torch.randn(64, 1, device=device)

model_ckpt = DeepNetCheckpointed().to(device)

for use_ckpt, label in [(False, "Normal      "), (True, "Checkpointed")]:
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    x_in = X_ckpt.clone().requires_grad_(True)
    out  = model_ckpt(x_in, use_ckpt=use_ckpt)
    nn.MSELoss()(out, y_ckpt).backward()
    if device.type == "cuda":
        mem = torch.cuda.max_memory_allocated() / 1e6
        print(f"{label}: peak GPU memory = {mem:.1f}MB")
    else:
        print(f"{label}: (memory measurement requires CUDA; forward/backward succeeded)")

print("On GPU: checkpointing saves 30-50% memory; use when OOM is the bottleneck.")
